
## 1. Catalyst Optimizer:


- **Catalyst Optimizer** is `Spark SQL's query optimizer`. 
- It improves the `Spark SQL's queries execution plan` to help `Spark SQL's queries run efficiently`.


### Solution 1: If employees_delta is a Delta table:


Use **spark.table()**

In [0]:
df = spark.table("workspace.default.employees_delta")

result = df.filter(df["salary"] > 30000).select("name", "salary")

result.show()


### Solution 2: If you want to read a CSV file:


Use the **actual CSV file path**:

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/data_volume/employees.csv",
    header=True,
    inferSchema=True
)

result = df.filter(df["salary"] > 50000).select("name", "salary")

result.show()


### Types of Catalyst Optimization:


- **1. Rule-Based Optimization (RBO)**
- **2. Cost-Based Optimization (CBO)**


## 1. Rule-Based Optimization (RBO):


- **Rule-Based Optimization** is the `predefined optimization rules`.
- It `improves spark sql query execution without comparing different execution costs`.
- Applies fixed `optimization rules`. 
- `Rule-driven` optimization.

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/data_volume/employees.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
display(df)

In [0]:
from pyspark.sql.functions import col, desc

result = (
    df.filter(col("salary") > 50000)
      .select("name", "salary")
      .orderBy(desc("salary"))
)

display(result)


### First create a temporary view:

In [0]:
df.createOrReplaceTempView("employees")


### Then execute:

In [0]:
%sql

SELECT name, salary
FROM employees
WHERE salary > 50000
ORDER BY salary DESC;


## 2. Cost-Based Optimization (CBO):


- **Cost-Based Optimization** uses `statistics of table`, such as `row counts and data sizes`, 
- It `improves spark sql query execution comparing different execution costs and choose a better plan`.
- Applies `fixed cost optimization`.
- `Cost driven` optimization.


## Difference between RBO and CBO in Spark SQL (Interview):


| `RBO (Rule-Based Optimizer)` | `CBO (Cost-Based Optimizer)` |
|---|---|
| Uses predefined optimization rules. | Uses statistics and cost estimates. |
| Does not comparing different execution costs. | Compares estimated execution costs. |
| Applies fixed optimization rules. | To estimate the cost of different execution plans and choose a better plan. |
| Rule-driven optimization. | Statistics-driven optimization. |


## 2. Tungsten:


**Tungsten** refers to `Spark SQL's query execution engine` that helps to `reduce memory overhead (RAM)` and `improve CPU efficiency`.

**Example:** 

**Spark can use efficient memory representations and code generation to execute supported operations more efficiently.**


## 3. Explain Plans:


- An **explain plan** means `explain how Spark plans to execute a query`. 
- It's operations such as `scans, filters, joins, exchanges, and shuffles`.

In [0]:
df = spark.read.csv("/Volumes/workspace/default/data_volume/employees.csv", header=True, inferSchema=True)

result = df.filter(df.salary > 50000)

result.explain(True)


## 4. Shuffle:


- **Shuffle** is the process of `redistributing data across different partitions or executors` in Spark.
- It is commonly occurs in `wide transformations`.
- Common Operations of shuffle are `groupBy(), distinct(), orderBy(), repartition(), Sort-based joins`.

In [0]:
result = df.groupBy("department").count()

display(result)


## 5. Data skew:


**Data skew** occurs when `data is distributed unevenly across partitions`.

**Example:**

**Suppose we have 1 million employee records distributed across 4 partitions.**

**Balanced data distribution:**

**Partition 1** - 250K records

**Partition 2** - 250K records

**Partition 3** - 250K records

**Partition 4** - 250K records


**Skewed data distribution:**

**Partition 1** - 100K records

**Partition 2** - 100K records

**Partition 3** - 100K records

**Partition 4** - 700K records



### Example:



**Consider an employee dataset with 10,000 records.**

| Department | Records |
|------------|--------:|
| IT         | 8,000   |
| HR         | 1,000   |
| Sales      | 500     |
| Finance    | 500     |
| **Total**  | **10,000** |

Here, IT contains most of the records.

If Spark distributes data by department, the partition processing IT records may receive much more data than the other partitions.


### How to identify Data Skew?

In [0]:
df.groupBy("department").count().display()


### handle Data Skew?

Data Skew is handle by `Salting, Broadcast Join, AQE, Repartitioning, Filtering`.


| Technique | Explanation |
|---|---|
| Salting | Adds `random values to skewed keys` to `distribute data evenly across partitions`. |
| Broadcast Join |` Broadcasts a small table to all worker nodes` to avoid shuffling a large table. |
| AQE | Dynamically handles `skewed partitions` and optimizes `query execution at runtime`. |
| Repartitioning | `Redistributes data across partitions` to `improve parallel processing`. |
| Filtering | `Removes unnecessary records before processing` to `reduce data volume`. |


### Real-time example:


**Suppose an e-commerce company joins orders and customers using customer_id.**

One customer has 1 million orders, while most customers have only 10–20 orders.

This creates data skew because one join key contains significantly more records than others.

Solution: Use `salting` to distribute the heavily repeated customer key across multiple partitions.


## 6. Salting:


- **Salting** is used to `reduce data skew by adding a random or calculated value` to `skewed join keys`. 
- Distributing the `data across multiple partitions`.

In [0]:
from pyspark.sql.functions import *

df = df.withColumn(
    "salt",
    floor(rand() * 5)
)


**This creates salt values from 0 to 4.**

For a skewed join, we can combine the original key with the salt value and replicate matching records on the smaller side of the join.


## 7. Broadcast Join:


A **broadcast join** is used to send a `small datasets to all executors` then Spark can join it with a `large dataset without shuffling the large dataset`.


### 1. Create employees DataFrame:

In [0]:
employees_data = [
    (1, "Arun Kumar", 55000, 101),
    (2, "Priya Sharma", 62000, 101),
    (3, "Karthik Raj", 48000, 102),
    (4, "Divya Lakshmi", 51000, 103),
    (5, "Suresh Babu", 45000, 104),
    (6, "Meena Iyer", 58000, 105),
    (7, "Rahul Verma", 67000, 101),
    (8, "Anitha Selvam", 47000, 102),
    (9, "Vignesh Kumar", 43000, 106),
    (10, "Deepa Nair", 53000, 103)
]

employees_columns = [
    "employee_id", "name", "salary", "department_id"
]

employees_df = spark.createDataFrame(
    employees_data, employees_columns
)

display(employees_df)


### 2. Create departments DataFrame:

In [0]:
departments_data = [
    (101, "Engineering"),
    (102, "Finance"),
    (103, "Human Resources"),
    (104, "Sales"),
    (105, "Marketing"),
    (106, "Operations")
]

departments_columns = ["department_id", "department_name"]

departments_df = spark.createDataFrame(
    departments_data, departments_columns
)

display(departments_df)


### 3. Perform Broadcast Join:

In [0]:
from pyspark.sql.functions import broadcast

result = employees_df.join(
    broadcast(departments_df),
    "department_id",
    "inner"
)

display(result,truncate=False)


## 8. Repartition vs Coalesce:


### Repartition:


- **repartition()** is used to `increases or decreases a partitions` by `redistributing data across partitions`. 
- Performs a `shuffle`.
- `More expensive`.
- Better for `redistributing data evenly`.	


### Coalesce:


- **coalesce()** is used to `decreases partitions`.
- Usually avoids a `full shuffle`.
- `Less expensive`.
- Better for `reducing the number of partitions`.


### Repartition Example:

In [0]:
df = spark.table("workspace.default.employees_delta")

repartitioned_df = df.repartition(4)

display(repartitioned_df)

print("Repartition completed successfully")


### Coalesce Example:

In [0]:
coalesced_df = df.coalesce(2)

display(coalesced_df)

print("Coalesce completed successfully")


### Compare partition counts without RDD:

In [0]:
from pyspark.sql.functions import spark_partition_id

print("Before repartition:")

df.select(
    spark_partition_id().alias("partition_id")
).distinct().orderBy("partition_id").show()

print("After repartition:")

repartitioned_df.select(
    spark_partition_id().alias("partition_id")
).distinct().orderBy("partition_id").show()


## 9. Caching:


- **Caching** is a process of `store frequently accessed DataFrame data in memory (RAM)` or `on disk`.
- using `df.cache()` in DataFrame API and `CACHE TABLE` in Spark SQL.


### df.cache() vs CACHE TABLE


| df.cache() | CACHE TABLE |
|---|---|
| Used with `DataFrame API`. | Used with `Spark SQL`. |
| Caches a `DataFrame`. | Caches a `table or view`. |
| Example: `df.cache()` | Example: `CACHE TABLE employees` |

In [0]:
df.cache()

df.count()

display(df)

In [0]:
%sql
CACHE TABLE employees;


## 10. Adaptive Query Execution (AQE):


- **Adaptive Query Execution (AQE)** is a `Spark SQL optimization feature`.
- It optimizes `the query execution plan at runtime` using collected `actual data statistics` during execution.


### Important AQE Configurations:


| Configuration | Purpose |
|---|---|
| `spark.sql.adaptive.enabled` | Enables Adaptive Query Execution (AQE). |
| `spark.sql.adaptive.coalescePartitions.enabled` | Enables automatic coalescing of shuffle partitions. |
| `spark.sql.adaptive.skewJoin.enabled` | Enables skew join optimization to handle uneven data distribution. |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | Controls the size threshold for adaptive broadcast joins at runtime. |

In [0]:
spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)


## 11. Small File Problem:


- The **small file problem** occurs when `a large number of small files are generated in storage`, 
- Increasing `metadata load` and reducing `Spark read operations.`



**Example:**

**Suppose a data pipeline generates 10,000 files of 1 MB each instead of 10 files of 1 GB each.**

Spark must handle many file metadata entries and file-reading operations, which can reduce performance.

How to solve it:
- Use **coalesce()** before writing when appropriate.
- Use **repartition()** to control output partitioning.
- Use **Delta Lake OPTIMIZE** for file compaction.


**Example:**

In [0]:
df.coalesce(5).write \
    .mode("overwrite") \
    .parquet("/data/employees")


**Delta Lake OPTIMIZE:**

In [0]:
OPTIMIZE employees;


**Interview point:** 

I solve the small file problem by **controlling output partitions and using Delta Lake OPTIMIZE** to compact small files into larger files.


### 12. Java and Kryo Serialization:


**Serialization** is the process of `converting objects into a byte streams` that can be `transferred over a network or stored`.


### Java Serialization:


- **Java Serialization** is the `default serialization mechanism` in Apache Spark. 
- That `converts Java objects into a byte stream`.
- It is `easy to use` and requires `minimal configuration`.
- It is `slower` than Kryo.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("JavaSerialization") \
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.JavaSerializer"
    ) \
    .getOrCreate()


### Kryo Serialization:


- **Kryo Serialization** is a `high-performance serialization mechanism` in Apache Spark.
- That converts `objects into a compact byte stream`.
- It is `faster` than Java Serialization in many workloads.
- It produces `smaller serialized data`.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KryoSerialization") \
    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    ) \
    .getOrCreate()


**Interview point:**

- `Kryo serialization is generally preferred over Java serialization`,
- When `serialization performance and memory efficiency` are important.


## 13. Encoders:


**Encoders** is used to convert `JVM objects into Spark's internal binary representation` and convert `them back when needed`.



**Why are Encoders used?**

- Convert `JVM objects into Spark's internal binary representation`.
- Convert `Spark's internal binary representation into JVM objects`.
- Spark understand `object schemas and data types`.


**What are the different types of Encoders in Spark?**

- **Primitive Encoder:** Used for `basic data types`.
- **Product Encoder**: Used for` Scala case classes`.
- **Bean Encoder:** Used for `Java Bean objects`.
- **Row Encoder:** Used for `Spark SQL Row-based data`.


## Interview Questions for Encoder:


**Q1. What is the main purpose of an Encoder?**

**Encoders** is used to convert `JVM objects into Spark's internal binary representation` and convert `them back when needed`.


**Q2. What is the difference between Encoder and Serializer?**

| Encoder | Serializer |
|---|---|
| Converts `JVM objects into Spark internal binary representation` | Converts `objects into byte streams` |
| Used in `Dataset processing` | Used for `object serialization and data transfer` |
| Understands `object schema and data types` | Converting `objects into serialized bytes` |
| Supports` Spark SQL optimizations` | Reduces `serialization and data transfer` overhead |
| Enables Spark SQL to process `structured object data` | Enables efficient `object storage and transfer` |


**Q3. Does PySpark support Encoders?**

- **No**, 
- PySpark `does not` support **explicit typed Encoders** like `Scala and Java`. 
- It primarily uses `DataFrames with schemas`.


**Q4. What is a Product Encoder?**

A **Product Encoder** converts `Scala case class objects into Spark's internal binary representation` and back.



**Q5. Are Encoders and Kryo the same?**

- No. 

- **Encoders** is used to convert `JVM objects into Spark's internal binary representation` and convert `them back when needed`.
- **Kryo Serialization** is the process of `converting objects into a compact byte streams`.


**Interview point:**

Encoders provide `efficient conversion between JVM objects and Spark's internal data format`, especially when working with typed Datasets.


## 14. Partition Pruning:


**Partition pruning** is an `optimization technique` in Spark that `reads only the required partitions` instead of `scanning all partitions`.


### How to Verify Partition Pruning?


Use `explain()` to inspect the Spark execution plan.


### Real-time example:


**Consider an e-commerce company storing 10 million orders partitioned by year and month.**

If we want to retrieve orders from January 2026, Spark reads only the relevant partition instead of scanning all available partitions.



### How Partition Pruning Works?


**Suppose the orders table has the following partitions:**


**orders/**

**│**

**├── year=2024/**

**│   ├── month=01/**

**│   └── month=02/**

**│**

**├── year=2025/**

**│   ├── month=01/**

**│   └── month=02/**

**│**

**└── year=2026/**

**|  ├── month=01/**

**|  └── month=02/**



### Creating Partitioned Data:

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PartitionPruning") \
    .getOrCreate()

data = [
    (101, "Franklin", 50000, 2025, 1),
    (102, "John", 60000, 2025, 2),
    (103, "David", 70000, 2026, 1),
    (104, "Smith", 80000, 2026, 2)
]

columns = [
    "employee_id",
    "name",
    "salary",
    "year",
    "month"
]

df = spark.createDataFrame(data, columns)

df.write \
    .partitionBy("year", "month") \
    .mode("overwrite") \
    .parquet("/tmp/employee_data")


### Applying Partition Pruning:

In [0]:
df = spark.read.parquet("/tmp/employee_data")

filtered_df = df.filter(
    (df.year == 2026) &
    (df.month == 1)
)

filtered_df.show()


### Important Interview Questions for Partition Pruning:


**Q1. What is Partition Pruning?**

**Partition pruning** is an `optimization technique` in Spark that `reads only the required partitions` instead of `scanning all partitions`.


**Q2. What is the main advantage of Partition Pruning?**

It `reduces all data scanning, disk I/O and query execution time`.


**Q3. Does Partition Pruning require partitioned data?**

- **Yes**, 
- It requires `data organized into partitions` with `filterable partition columns`.


**Q4. How do you verify Partition Pruning?**

By using `DataFrame.explain(True)` and `checking the PartitionFilters` section.


**Q5. What happens if we filter using a non-partition column?**

- **Spark** cannot eliminate `partitions using that filter` alone. 
- It may use other `optimizations` such as `Parquet predicate pushdown or data skipping`.


**Q6. Is Partition Pruning the same as Predicate Pushdown?**

- **No**,**Partition Pruning** the `not` same as Predicate Pushdown.
- **Partition pruning** is an `optimization technique` in Spark that `reads only the required partitions` instead of `scanning all partitions`.
- **Predicate Pushdown** is a `query optimization technique` in Spark that `pushes filter conditions closer to the data source`, reducing the `amount of unnecessary data read` and improving `query performance`.


## 15. Partitioning:


- **Partitioning** means dividing `large data into smaller parts`, Spark can process them in `parallel`.
- This can helps spark `to skips unnecessary data` during queries.


### Why is Partitioning Used?

- Enables `parallel processing`.
- Distributes `workloads across executors`.
- Improves `processing performance`.
- Helps `manage large datasets`.


### Types of Partitioning:


| Partitioning Type | Explanation | Real-Time Example |
|---|---|---|
| Hash Partitioning | Distributes records into partitions based on the hash value of a column. | Distributing employee records based on EmployeeID. |
| Range Partitioning | Distributes records based on ranges of column values. | Dividing employee salaries into different ranges. |
| Round Robin Partitioning | Distributes records approximately evenly across partitions without considering column values. | Distributing 10 lakh records evenly across 10 partitions. |
| Custom Partitioning | user-defined logic how records are distributed across partitions. | Distributing customer records based on custom business rules. |


### Real-time example:

**Suppose an e-commerce company has customer order data.**

In [0]:
data = [
    (101, "Chennai", 5000),
    (102, "Bangalore", 7000),
    (103, "Chennai", 3000),
    (104, "Hyderabad", 6000),
    (105, "Bangalore", 8000)
]

columns = ["order_id", "city", "amount"]

df = spark.createDataFrame(data, columns)


**Check the number of partitions:**

In [0]:
df.rdd.getNumPartitions()


### Important Interview Questions:


**Q1. What is the default number of partitions in Spark?**

- The **default number of partitions in Spark** depends on the `input source, configuration and execution plan`. 
- For **shuffle operations**, Spark SQL commonly uses `spark.sql.shuffle.partitions`, which default is `200` in **standard Spark configurations**.


**Q2. Can repartition increase the number of partitions?**

- **Yes**. 
- `repartition()` can `increase or decrease` partitions by `redistributing data across partitions`.


**Q3. Can coalesce increase partitions?**

**No.**
`coalesce()` is primarily used to `reduce` the number of partitions.


**Q4. Why does repartition perform a shuffle?**

Because Spark **redistributes data across partitions** to `achieve the requested partition count or column-based distribution`.


**Q5. When should we use coalesce?**

- When `reducing the number of partitions` before `writing output files`, 
- especially when `redistributing data across partitions` is unnecessary.


**Q6. Does more partitioning always improve performance?**

- **No.** 
- Too many `small partitions create scheduling overhead` and can produce `too many small files`.


**Q7. What is data skew?**

**Data skew** occurs when `data is distributed unevenly across partitions`.

**Example:**

**Suppose we have 1 million employee records distributed across 4 partitions.**

**Balanced data distribution:**

**Partition 1** - 250K records

**Partition 2** - 250K records

**Partition 3** - 250K records

**Partition 4** - 250K records


**Skewed data distribution:**

**Partition 1** - 100K records

**Partition 2** - 100K records

**Partition 3** - 100K records

**Partition 4** - 700K records



## 16. OPTIMIZE and Compaction in Delta Lake:


**OPTIMIZE** is a `Delta Lake command` used to `compacting multiple small files into fewer larger files` and improve `query performance`.


**Compaction** is used to `combines multiple small files into fewer larger files` and improve `storage and query performance`.


## Why do we need OPTIMIZE?


In **Delta Lake**, frequent `INSERT, UPDATE, DELETE and MERGE operations` can create many small files.


### OPTIMIZE Command:

In [0]:
OPTIMIZE sales;


### Compaction for a particular partition:

In [0]:
OPTIMIZE sales
WHERE year = 2026 AND month = 10;


### Important Interview Questions for OPTIMIZE and Compaction in Delta Lake:


**Q1. What is the small-file problem?**

- The **small file problem** occurs when `a large number of small files are generated in storage`, 
- Increasing `metadata load` and reducing `Spark read operations.`


**Q2. How do you solve the small-file problem in Delta Lake?**

We can use the `OPTIMIZE` command to **compact multiple small files into fewer larger files**.


**Q3. Does OPTIMIZE delete old files immediately?**

- **No**,OPTIMIZE **does not** immediately delete old data files. 
- It creates `new optimized files` and marks the `old files as obsolete` in the **Delta transaction log**.


**Q4. What is the default target file size for OPTIMIZE?**

- **Delta Lake** targets approximately `1 GB files by default`.
- target file can be `configured` and actual file `sizes vary`.


**Q5. Can we run OPTIMIZE on a specific partition?**

- **Yes**,we can run OPTIMIZE on a specific partition.
- We can use a `WHERE` condition to **optimize selected partitions**.

**Example:**

OPTIMIZE sales

WHERE year = 2026 AND month = 10;


**Q6. Does OPTIMIZE change the table data?**

- **No**, OPTIMIZE **does not** change the logical table data.
- It `reorganizes the physical data files` without `changing the logical table contents`.


**Q7. What is the difference between OPTIMIZE and ZORDER?**

- **OPTIMIZE** `compacts and reorganizes` data files.
- **ZORDER** improves `data layout for specified columns` to help data skipping.
- **ZORDER** is an `additional optimization` used with OPTIMIZE.


## 17. Broadcast Hint in DataFrame API vs SQL:


- A **Broadcast Hint** in Apache Spark is a `query optimization technique`.
- That tells Spark to broadcast a `small table to all worker nodes` so that it can perform a `Broadcast Hash Join` with a **large table**.
- It avoids `shuffling the large table` across the cluster and can `improve join performance`.


## Broadcast Hint in DataFrame API:


In DataFrame API, we use the `broadcast()` function to **broadcast a smaller DataFrame during a join**.


### Syntax:

In [0]:
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "department_id",
    "inner"
)


**Example:**

In [0]:
from pyspark.sql.functions import broadcast

result = employees_df.join(
    broadcast(departments_df),
    "department_id",
    "inner"
)

result.show()


## Broadcast Hint in Spark SQL:


In Spark SQL, we use the `BROADCAST` hint to **broadcast a smaller table during a join**.


### Syntax:

In [0]:
%sql
SELECT /*+ BROADCAST(small_table) */
    *
FROM large_table
JOIN small_table
ON large_table.department_id = small_table.department_id;


### Example:


### First, create temporary views:

In [0]:
employees_df.createOrReplaceTempView("employees")

departments_df.createOrReplaceTempView("departments")


### Execute Spark SQL:

In [0]:
%sql

SELECT /*+ BROADCAST(d) */
    e.employee_id,
    e.name,
    e.salary,
    d.department_name
FROM employees e
JOIN departments d
ON e.department_id = d.department_id;


### Important Interview Questions for Broadcast Hint in DataFrame API vs SQL:


**Q1. Why do we use Broadcast Hint?**

We use Broadcast Hint to `Broadcast a small table to all executors` and avoid `shuffling the large table during a join`.


**Q2. What is the difference between Broadcast Join and Shuffle Join?**

| Broadcast Join | Shuffle Join |
|---|---|
| Broadcasts the small table to all worker nodes. | Redistributes data across partitions using a shuffle operation. |
| Avoids shuffling the large table. | Usually shuffles both sides in a Sort Merge Join. |
| Suitable for joining a small table with a large table. | Suitable for joining two large tables. |
| Commonly uses Broadcast Hash Join. | Commonly uses Sort Merge Join. |


**Q3. What happens if we broadcast a very large table?**

- It can cause `high memory consumption, excessive network transfer and executor memory issues`. 
- In extreme cases, it may cause `out-of-memory` errors.


**Q4. Is Broadcast Hint supported in both DataFrame API and SQL?**

- **Yes**.
- DataFrame API uses `broadcast()` and SQL uses the `BROADCAST` hint.


**Q5. What is the default automatic broadcast threshold?**

- In standard **Spark SQL** configurations, the default `spark.sql.autoBroadcastJoinThreshold is 10 MB`. 
- Databricks or other environments may `configure it differently`.


**Q6. Does Broadcast Hint guarantee a broadcast join?**

- **No**. 
- A **Broadcast Hint** strongly encourages `Spark to use a Broadcast Join`, but it **does not guarantee**.